In [13]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler


In [14]:
def run_ids2025_feature_engineering(file_path):
    print("[*] Launching IDS2025 Feature Engineering Pipeline.....")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Target dataset '{file_path}' not found in local directory.")
    # Read dataset in chunks or completely depending on your system RAM capacity
    print("[*] Ingesting CSV rows...")
    df = pd.read_csv(file_path)
    print(f"[+] Data loaded. Dimensions: {df.shape[0]} rows, {df.shape[1]} columns.") 

    # STEP 1: CLEAN MATHEMATICAL ANOMALIES (Inf / NaN)
    print("[*] Cleaning network flow math calc limits (Infinities/Nulls)...")
    #replace standard float infinities with NaN, then drop or fill them
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    missing_count = df.isnull().sum().sum()
    if missing_count > 0:
        print(f"[!] Warning: Found {missing_count} total missing/infinite data blocks. Clearing records...")
        df.dropna(inplace=True)

    # STEP 2: ISOLATE TARGET LABELS & DROP SOURCE PORT/IPs
    target_col = 'newLabel'
    if target_col not in df.columns:
        raise ValueError(f"Critical target columns '{target_col}' missing from your dataset layout.")
    y_raw = df[target_col]
    
    # Drop features that cause severe overfitting or data leakage
    # Source port shifts randomly; raw IPs or uniform timestamps must not enter tree node evaluations.
    features_to_drop = [target_col, 'Source Port']
    # If your dataset contains raw IP strings like 'Source IP', drop them here too:
    for col in ['Source IP', 'Dest IP', 'Timestamp']:
        if col in df.columns:
            features_to_drop.append(col)
            
    X = df.drop(columns=features_to_drop)
    
    # ----------------------------------------------------
    # STEP 3: BEHAVIORAL PORT BINNING (Destination Ports)
    # ----------------------------------------------------
    print("[*] Engineering Destination Port network profiles...")
    if 'Destination Port' in X.columns:
        # Categorize ports based on standard IANA rules
        X['PORT_WELL_KNOWN'] = (X['Destination Port'] < 1024).astype(int)
        X['PORT_REGISTERED'] = ((X['Destination Port'] >= 1024) & (X['Destination Port'] < 49152)).astype(int)
        X['PORT_DYNAMIC'] = (X['Destination Port'] >= 49152).astype(int)
        # Drop the raw Destination Port column so the model learns behavioral groupings instead of raw numbers
        X.drop(columns=['Destination Port'], inplace=True)

    # ----------------------------------------------------
    # STEP 4: LOG TRANSFORM DISTRIBUTIONS FOR VOLUMETRIC DATA
    # ----------------------------------------------------
    print("[*] Applying log scale transformation on skewed volumetric parameters...")
    # These exact headers account for the double-s spellings found in your file
    skewed_network_features = [
        'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
        'Flow Bytess', 'Flow Packetss', 'Fwd Packetss', 'Bwd Packetss',
        'Fwd Packet Length Max', 'Bwd Packet Length Max',
        'Fwd Packet Length Mean', 'Bwd Packet Length Mean'
    ]
    
    for feature in skewed_network_features:
        if feature in X.columns:
            # Use log1p (log(x + 1)) to safely compute records containing 0 bytes
            X[feature] = np.log1p(X[feature])

    # ----------------------------------------------------
    # STEP 5: CATEGORICAL PROTOCOL ENCODING
    # ----------------------------------------------------
    # Map Protocol (TCP=6, UDP=17, ICMP=1) cleanly
    if 'Protocol' in X.columns:
        if X['Protocol'].dtype == 'object':
            print("[*] Encoding text-based Protocol fields...")
            proto_encoder = LabelEncoder()
            X['Protocol'] = proto_encoder.fit_transform(X['Protocol'].astype(str))
            joblib.dump(proto_encoder, "protocol_encoder.pkl")

    # ----------------------------------------------------
    # STEP 6: MATRIX VECTOR COMPILATION & TARGET ENCODING
    # ----------------------------------------------------
    print("[*] Encapsulating threat output class matrices...")
    target_encoder = LabelEncoder()
    y_encoded = target_encoder.fit_transform(y_raw)
    
    print(f"[+] Map Configuration Classes Verified: {dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))}")
    
    return X, y_encoded, target_encoder

if __name__ == "__main__":
    # Target dataset file
    csv_filename = "../Data/raw/IDS2025.csv"
    
    try:
        X_engineered, y_encoded, target_encoder = run_ids2025_feature_engineering(csv_filename)
        
        # Split Data into Training and Validation Arrays (80/20)
        print("[*] Splitting dataset into training and validation environments...")
        X_train, X_test, y_train, y_test = train_test_split(
            X_engineered, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
        )
        
        # Scale remaining continuous numerical distributions uniformly
        print("[*] Normalizing continuous features with StandardScaler...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Save structural weights for deployment to the live model folders
        print("[*] Serializing engineered scaling artifacts...")
        joblib.dump(scaler, "robust_scaler.pkl")
        joblib.dump(target_encoder, "attack_classes.pkl")
        
        # Convert to compressed numpy binaries for rapid model execution
        np.save('X_train_processed.npy', X_train_scaled)
        np.save('X_test_processed.npy', X_test_scaled)
        np.save('y_train.npy', y_train)
        np.save('y_test.npy', y_test)
        
        print("\n" + "="*40)
        print("[+] SUCCESS: Feature Engineering Pipeline Completed.")
        print(f"    Processed Features Shape: {X_train_scaled.shape}")
        print("    Saved Artifacts: robust_scaler.pkl, attack_classes.pkl")
        print("    Saved Data Arrays: X_train_processed.npy, X_test_processed.npy")
        print("="*40)
        
    except Exception as e:
        print(f"[-] Pipeline aborted due to execution error: {str(e)}")

[*] Launching IDS2025 Feature Engineering Pipeline.....
[*] Ingesting CSV rows...
[+] Data loaded. Dimensions: 91830 rows, 80 columns.
[*] Cleaning network flow math calc limits (Infinities/Nulls)...
[!] Warning: Found 254 total missing/infinite data blocks. Clearing records...
[*] Engineering Destination Port network profiles...
[*] Applying log scale transformation on skewed volumetric parameters...
[*] Encapsulating threat output class matrices...
[+] Map Configuration Classes Verified: {'Botnet ARES': np.int64(0), 'Brute Force': np.int64(1), 'Dos/DDos': np.int64(2), 'Infiltration': np.int64(3), 'Normal': np.int64(4), 'PortScan': np.int64(5), 'Web Attack': np.int64(6)}
[*] Splitting dataset into training and validation environments...


C:\Users\Lenovo\AppData\Roaming\Python\Python314\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


[*] Normalizing continuous features with StandardScaler...
[*] Serializing engineered scaling artifacts...

[+] SUCCESS: Feature Engineering Pipeline Completed.
    Processed Features Shape: (73362, 80)
    Saved Artifacts: robust_scaler.pkl, attack_classes.pkl
    Saved Data Arrays: X_train_processed.npy, X_test_processed.npy
